In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import csv

def scrape_imdb_full_data():
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(options=options)

    url = "https://www.imdb.com/search/title/?title_type=feature&sort=user_rating,desc"
    
    driver.get(url)
    time.sleep(4)

    filename = "imdb_movies_dataset.csv"

    headers = [
        "IMDb_ID","Title","Year","Runtime","Certificate",
        "Rating","Vote_Count","Plot_Summary","IMDb_Link","Poster_URL"
    ]

    # create CSV and write header
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(headers)

    target_clicks = 200
    scraped_ids = set()

    for i in range(target_clicks):

        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

            wait = WebDriverWait(driver, 5)
            load_more_btn = wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "button.ipc-see-more__button"))
            )

            driver.execute_script("arguments[0].click();", load_more_btn)
            print(f"Clicked Load More {i+1}/{target_clicks}")
            time.sleep(3)

        except:
            print("No more Load More button")
            break

        # ---------- EXTRACT DATA AFTER EACH CLICK ----------
        soup = BeautifulSoup(driver.page_source, "html.parser")
        movies = soup.select("li.ipc-metadata-list-summary-item")

        new_rows = []

        for movie in movies:
            try:
                title_tag = movie.select_one("h3.ipc-title__text")
                title = title_tag.text.strip() if title_tag else "N/A"
                if ". " in title:
                    title = title.split(". ",1)[-1]

                meta = movie.select("span.dli-title-metadata-item")
                year = meta[0].text.strip() if len(meta)>0 else "N/A"
                runtime = meta[1].text.strip() if len(meta)>1 else "N/A"
                certificate = meta[2].text.strip() if len(meta)>2 else "N/A"

                rating_tag = movie.select_one("span.ipc-rating-star")
                rating = rating_tag.text.strip().split(" ")[0] if rating_tag else "N/A"

                plot_tag = movie.select_one("div.ipc-html-content-inner-div")
                plot = plot_tag.text.strip() if plot_tag else "N/A"

                link_tag = movie.select_one("a.ipc-title-link-wrapper")

                if link_tag:
                    href = link_tag.get("href")
                    imdb_link = f"https://www.imdb.com{href}"
                    imdb_id = href.split("/")[2]
                else:
                    imdb_link = "N/A"
                    imdb_id = "N/A"

                if imdb_id in scraped_ids:
                    continue

                scraped_ids.add(imdb_id)

                img_tag = movie.select_one("img.ipc-image")
                poster = img_tag.get("src") if img_tag else "N/A"

                new_rows.append([
                    imdb_id,title,year,runtime,certificate,
                    rating,"N/A",plot,imdb_link,poster
                ])

            except:
                continue

        # append new data to CSV
        with open(filename,"a",newline="",encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerows(new_rows)

        print(f"Saved {len(new_rows)} new movies to CSV")

    driver.quit()

if __name__ == "__main__":
    scrape_imdb_full_data()

Clicked Load More 1/200
Saved 100 new movies to CSV
Clicked Load More 2/200
Saved 50 new movies to CSV
Clicked Load More 3/200
Saved 50 new movies to CSV
Clicked Load More 4/200
Saved 50 new movies to CSV
Clicked Load More 5/200
Saved 50 new movies to CSV
Clicked Load More 6/200
Saved 50 new movies to CSV
Clicked Load More 7/200
Saved 50 new movies to CSV
Clicked Load More 8/200
Saved 50 new movies to CSV
Clicked Load More 9/200
Saved 50 new movies to CSV
Clicked Load More 10/200
Saved 50 new movies to CSV
Clicked Load More 11/200
Saved 50 new movies to CSV
Clicked Load More 12/200
Saved 50 new movies to CSV
Clicked Load More 13/200
Saved 50 new movies to CSV
Clicked Load More 14/200
Saved 50 new movies to CSV
Clicked Load More 15/200
Saved 50 new movies to CSV
Clicked Load More 16/200
Saved 50 new movies to CSV
Clicked Load More 17/200
Saved 0 new movies to CSV
Clicked Load More 18/200
Saved 100 new movies to CSV
Clicked Load More 19/200
Saved 0 new movies to CSV
Clicked Load More 20/

In [16]:
import pandas as pd
df = pd.read_csv("imdb_movies_dataset.csv")
df.head()

,IMDb_ID,Title,Year,Runtime,Certificate,Rating,Vote_Count,Plot_Summary,IMDb_Link,Poster_URL
0,tt37161609,Sky,2026,2h 14m,NaN,10 (1.1K),NaN,Modern relationships intertwine with career am...,https://www.imdb.com/title/tt37161609/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BMGMyMD...
1,tt40326325,Karanlik Sokaklar,2026,NaN,NaN,10 (5),NaN,NaN,https://www.imdb.com/title/tt40326325/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BMWMwYm...
2,tt40415811,Saravá Shalom,2026,1h 22m,NaN,10 (6),NaN,A narrative that shows the dialogue between Af...,https://www.imdb.com/title/tt40415811/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BMjk4OG...
3,tt37599251,Big Mountain Soul: Ski Africa,2025,45m,NaN,10 (15),NaN,Join the Big Mountain Soul group on an extraor...,https://www.imdb.com/title/tt37599251/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BOWJiZG...
4,tt39455347,TT for Life,2026,1h 27m,NaN,10 (6),NaN,"TT FOR LIFE plunges into the Isle of Man TT, t...",https://www.imdb.com/title/tt39455347/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BOGUzNj...


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   IMDb_ID       10000 non-null  object 
 1   Title         10000 non-null  object 
 2   Year          9999 non-null   object 
 3   Runtime       7914 non-null   object 
 4   Certificate   778 non-null    object 
 5   Rating        10000 non-null  object 
 6   Vote_Count    0 non-null      float64
 7   Plot_Summary  8406 non-null   object 
 8   IMDb_Link     10000 non-null  object 
 9   Poster_URL    9178 non-null   object 
dtypes: float64(1), object(9)
memory usage: 781.4+ KB


In [18]:
df.drop_duplicates(subset=["IMDb_ID"], inplace=True)

In [19]:
df.drop(columns=["Vote_Count"], inplace=True)

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   IMDb_ID       10000 non-null  object
 1   Title         10000 non-null  object
 2   Year          9999 non-null   object
 3   Runtime       7914 non-null   object
 4   Certificate   778 non-null    object
 5   Rating        10000 non-null  object
 6   Plot_Summary  8406 non-null   object
 7   IMDb_Link     10000 non-null  object
 8   Poster_URL    9178 non-null   object
dtypes: object(9)
memory usage: 703.3+ KB


In [21]:
df.head()

,IMDb_ID,Title,Year,Runtime,Certificate,Rating,Plot_Summary,IMDb_Link,Poster_URL
0,tt37161609,Sky,2026,2h 14m,NaN,10 (1.1K),Modern relationships intertwine with career am...,https://www.imdb.com/title/tt37161609/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BMGMyMD...
1,tt40326325,Karanlik Sokaklar,2026,NaN,NaN,10 (5),NaN,https://www.imdb.com/title/tt40326325/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BMWMwYm...
2,tt40415811,Saravá Shalom,2026,1h 22m,NaN,10 (6),A narrative that shows the dialogue between Af...,https://www.imdb.com/title/tt40415811/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BMjk4OG...
3,tt37599251,Big Mountain Soul: Ski Africa,2025,45m,NaN,10 (15),Join the Big Mountain Soul group on an extraor...,https://www.imdb.com/title/tt37599251/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BOWJiZG...
4,tt39455347,TT for Life,2026,1h 27m,NaN,10 (6),"TT FOR LIFE plunges into the Isle of Man TT, t...",https://www.imdb.com/title/tt39455347/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BOGUzNj...


In [22]:
df["Rating"]

0       10 (1.1K)
1          10 (5)
2          10 (6)
3         10 (15)
4          10 (6)
          ...    
9995     8.6 (42)
9996     8.6 (41)
9997     8.6 (19)
9998     8.6 (15)
9999     8.6 (12)
Name: Rating, Length: 10000, dtype: object

In [ ]:
# 1. Extract the rating and the count from the brackets
df[['Rating_Value', 'Count']] = df['Rating'].str.extract(r'([\d.]+)\s\((.*)\)')

# 2. Clean the 'Count' column:
# Handle 'K' (e3) and 'M' (e6), remove commas if any, and convert to float
df['Count_Numeric'] = (
    df['Count']
    .replace({'K': 'e3', 'M': 'e6', ',': ''}, regex=True)
    .astype(float)
)

# 3. Sort the DataFrame by the numeric count (e.g., descending order)
df_sorted = df.sort_values(by='Count_Numeric', ascending=False)

# Optional: Convert Rating_Value to float for proper numeric sorting/math
df_sorted['Rating_Value'] = df_sorted['Rating_Value'].astype(float)


        IMDb_ID                     Title  Year Runtime Certificate  \
1450  tt0111161  The Shawshank Redemption  1994  2h 22m           R   
2838  tt0468569           The Dark Knight  2008  2h 32m       PG-13   
6021  tt1375666                 Inception  2010  2h 28m       PG-13   
6022  tt0137523                Fight Club  1999  2h 19m           R   
6023  tt0109830              Forrest Gump  1994  2h 22m       PG-13   

          Rating                                       Plot_Summary  \
1450  9.3 (3.2M)  A banker convicted of uxoricide forms a friend...   
2838  9.1 (3.1M)  When a menace known as the Joker wreaks havoc ...   
6021  8.8 (2.8M)  A thief who steals corporate secrets through t...   
6022  8.8 (2.6M)  An insomniac office worker and a devil-may-car...   
6023  8.8 (2.5M)  The history of the United States from the 1950...   

                                              IMDb_Link  \
1450  https://www.imdb.com/title/tt0111161/?ref_=sr_...   
2838  https://www.imdb.com/t

In [24]:
df_sorted.head()

,IMDb_ID,Title,Year,Runtime,Certificate,Rating,Plot_Summary,IMDb_Link,Poster_URL,Rating_Value,Count,Count_Numeric
1450,tt0111161,The Shawshank Redemption,1994,2h 22m,R,9.3 (3.2M),A banker convicted of uxoricide forms a friend...,https://www.imdb.com/title/tt0111161/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMDAyY2...,9.3,3.2M,3200000.0
2838,tt0468569,The Dark Knight,2008,2h 32m,PG-13,9.1 (3.1M),When a menace known as the Joker wreaks havoc ...,https://www.imdb.com/title/tt0468569/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMTMxNT...,9.1,3.1M,3100000.0
6021,tt1375666,Inception,2010,2h 28m,PG-13,8.8 (2.8M),A thief who steals corporate secrets through t...,https://www.imdb.com/title/tt1375666/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMjAxMz...,8.8,2.8M,2800000.0
6022,tt0137523,Fight Club,1999,2h 19m,R,8.8 (2.6M),An insomniac office worker and a devil-may-car...,https://www.imdb.com/title/tt0137523/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BOTgyOG...,8.8,2.6M,2600000.0
6023,tt0109830,Forrest Gump,1994,2h 22m,PG-13,8.8 (2.5M),The history of the United States from the 1950...,https://www.imdb.com/title/tt0109830/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BNDYwNz...,8.8,2.5M,2500000.0


In [25]:
df_sorted.drop(columns=["Rating","Count_Numeric"], inplace=True)

In [26]:
df_sorted.head()

,IMDb_ID,Title,Year,Runtime,Certificate,Plot_Summary,IMDb_Link,Poster_URL,Rating_Value,Count
1450,tt0111161,The Shawshank Redemption,1994,2h 22m,R,A banker convicted of uxoricide forms a friend...,https://www.imdb.com/title/tt0111161/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMDAyY2...,9.3,3.2M
2838,tt0468569,The Dark Knight,2008,2h 32m,PG-13,When a menace known as the Joker wreaks havoc ...,https://www.imdb.com/title/tt0468569/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMTMxNT...,9.1,3.1M
6021,tt1375666,Inception,2010,2h 28m,PG-13,A thief who steals corporate secrets through t...,https://www.imdb.com/title/tt1375666/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMjAxMz...,8.8,2.8M
6022,tt0137523,Fight Club,1999,2h 19m,R,An insomniac office worker and a devil-may-car...,https://www.imdb.com/title/tt0137523/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BOTgyOG...,8.8,2.6M
6023,tt0109830,Forrest Gump,1994,2h 22m,PG-13,The history of the United States from the 1950...,https://www.imdb.com/title/tt0109830/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BNDYwNz...,8.8,2.5M


In [27]:
df_sorted = df_sorted[["IMDb_ID","Title","Year","Runtime","Rating_Value","Count","Certificate","Plot_Summary","IMDb_Link","Poster_URL"]]

In [28]:
df_sorted.head()

,IMDb_ID,Title,Year,Runtime,Rating_Value,Count,Certificate,Plot_Summary,IMDb_Link,Poster_URL
1450,tt0111161,The Shawshank Redemption,1994,2h 22m,9.3,3.2M,R,A banker convicted of uxoricide forms a friend...,https://www.imdb.com/title/tt0111161/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMDAyY2...
2838,tt0468569,The Dark Knight,2008,2h 32m,9.1,3.1M,PG-13,When a menace known as the Joker wreaks havoc ...,https://www.imdb.com/title/tt0468569/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMTMxNT...
6021,tt1375666,Inception,2010,2h 28m,8.8,2.8M,PG-13,A thief who steals corporate secrets through t...,https://www.imdb.com/title/tt1375666/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMjAxMz...
6022,tt0137523,Fight Club,1999,2h 19m,8.8,2.6M,R,An insomniac office worker and a devil-may-car...,https://www.imdb.com/title/tt0137523/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BOTgyOG...
6023,tt0109830,Forrest Gump,1994,2h 22m,8.8,2.5M,PG-13,The history of the United States from the 1950...,https://www.imdb.com/title/tt0109830/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BNDYwNz...


In [29]:
df_sorted.rename(columns={"Count":"Rating_Count"}, inplace=True)

In [30]:
df_sorted

,IMDb_ID,Title,Year,Runtime,Rating_Value,Rating_Count,Certificate,Plot_Summary,IMDb_Link,Poster_URL
1450,tt0111161,The Shawshank Redemption,1994,2h 22m,9.3,3.2M,R,A banker convicted of uxoricide forms a friend...,https://www.imdb.com/title/tt0111161/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMDAyY2...
2838,tt0468569,The Dark Knight,2008,2h 32m,9.1,3.1M,PG-13,When a menace known as the Joker wreaks havoc ...,https://www.imdb.com/title/tt0468569/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMTMxNT...
6021,tt1375666,Inception,2010,2h 28m,8.8,2.8M,PG-13,A thief who steals corporate secrets through t...,https://www.imdb.com/title/tt1375666/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BMjAxMz...
6022,tt0137523,Fight Club,1999,2h 19m,8.8,2.6M,R,An insomniac office worker and a devil-may-car...,https://www.imdb.com/title/tt0137523/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BOTgyOG...
6023,tt0109830,Forrest Gump,1994,2h 22m,8.8,2.5M,PG-13,The history of the United States from the 1950...,https://www.imdb.com/title/tt0109830/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BNDYwNz...
...,...,...,...,...,...,...,...,...,...,...
6766,tt34376717,Reinosa 1987: El precio de la reconversión ind...,2025,NaN,8.8,5,NaN,NaN,https://www.imdb.com/title/tt34376717/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BNjU1MW...
4221,tt3279992,An Anarchist Life,2014,52m,9.0,5,NaN,"It's a model story, an extraordinary adventure...",https://www.imdb.com/title/tt3279992/?ref_=sr_...,https://m.media-amazon.com/images/M/MV5BNTgzM2...
6843,tt0172087,A cui e vina?,1965,NaN,8.8,5,NaN,NaN,https://www.imdb.com/title/tt0172087/?ref_=sr_...,NaN
75,tt38628151,Children of Yamato,2025,1h 20m,9.8,5,NaN,The U.S. post-WWII reconstruction of Japan is ...,https://www.imdb.com/title/tt38628151/?ref_=sr...,https://m.media-amazon.com/images/M/MV5BN2I2Nj...
